# B7 — Industrialisation du pipeline vision : du notebook au script rejouable

**Périmètre de ce TP** : le module B7 couvre deux chantiers sur deux projets distincts —
l'optimisation Optuna et l'explicabilité SHAP sur le modèle tabulaire de maintenance
(projet `ML/`, autre venv, autre dataset), et l'industrialisation du pipeline vision (DL).
**Ce TP7 traite uniquement le second volet.**

> *Product Owner : plus personne n'arrive à rejouer vos résultats. Tout vit dans des
> notebooks, les chemins sont codés en dur, l'ordre des cellules change les scores…*

TP1 à TP6 ont exploré, entraîné et mesuré — en notebook. Ce TP7 ne ré-explore rien :
il **sort la logique du notebook** et prouve qu'une seule commande rejoue tout
l'enchaînement `data → modèle → entraînement → CodeCarbon → évaluation → figures`.

| § | Contenu |
|---|---|
| §1 | La règle des trois couches — état des lieux TP1-TP6 |
| §2 | `config.py` — centraliser les chemins, plus un seul codé en dur |
| §3 | `scripts/run_vision_pipeline.py` — le pipeline CLI |
| §4 | Exécuter le pipeline en une commande |
| §5 | Idempotence — deux exécutions, un seul score ? |
| §6 | Synthèse et périmètre restant (ML — hors TP7) |


## §1 — La règle des trois couches

```
notebook.ipynb        ← explore, raconte, visualise — importe la logique, ne la contient pas
src/indusense/vision/ ← logique réutilisable, testable, sans chemin en dur
scripts/run_*.py      ← orchestre les briques src/ en une commande CLI rejouable
```

**État des lieux après TP1-TP6** : la couche `src/` existe déjà et est plutôt saine —
`dataset.py`, `augment.py`, `model.py`, `train.py`, `anomaly.py` contiennent des fonctions
pures (pas de `print` décoratif, pas de chemin en dur, tout en paramètres). Les notebooks
TP1-TP6 **importent** ces modules plutôt que de dupliquer la logique.

**Deux écarts identifiés :**

| Écart | Où | Impact |
|---|---|---|
| Chemins codés en dur | Chaque notebook redéfinit `Path("bottle")`, `"checkpoints/ae_v3_best.keras"`, `"figures/..."` | Fonctionne seulement si on lance depuis `DL/` — fragile en CI ou sur une autre machine |
| Logique inline non descendue | TP5 §1 : le scorer SSE (`keras.Model` avec `Subtract`+`Lambda`) est construit en cellule, pas en fonction `src/` | Pas testable, pas réutilisable hors de ce notebook |

Ce TP7 corrige le premier écart (`config.py`, §2) et referme la boucle avec un
point d'entrée unique (`scripts/run_vision_pipeline.py`, §3). Le second écart (scorer SHAP)
reste un refactor identifié mais non traité ici — hors périmètre de l'industrialisation
du pipeline d'entraînement/évaluation.

## §2 — `config.py` : centraliser les chemins

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
import config

for name in ["BOTTLE_ROOT", "CHECKPOINTS_DIR", "FIGURES_DIR", "EMISSIONS_DIR", "REPORTS_DIR"]:
    p = getattr(config, name)
    print(f"{name:16s} = {p}  (existe : {p.exists()})")

print(f"\nRANDOM_SEED       = {config.RANDOM_SEED}")
print(f"COUNTRY_ISO_CODE  = {config.COUNTRY_ISO_CODE}  (forcé — pas de géoloc IP, reproductible en CI)")
print(f"DEFAULT_FILTERS   = {config.DEFAULT_FILTERS}")


BOTTLE_ROOT      = C:\Users\Aelion\py-init\DL\bottle  (existe : True)
CHECKPOINTS_DIR  = C:\Users\Aelion\py-init\DL\checkpoints  (existe : True)
FIGURES_DIR      = C:\Users\Aelion\py-init\DL\figures  (existe : True)
EMISSIONS_DIR    = C:\Users\Aelion\py-init\DL\emissions  (existe : True)
REPORTS_DIR      = C:\Users\Aelion\py-init\DL\reports  (existe : True)

RANDOM_SEED       = 42
COUNTRY_ISO_CODE  = FRA  (forcé — pas de géoloc IP, reproductible en CI)
DEFAULT_FILTERS   = (32, 64, 128, 64)


Plus aucun script ni notebook n'a besoin de retaper `Path("bottle")` : tout passe par
`config.BOTTLE_ROOT`. Un changement de structure de dossiers se fait à un seul endroit.

## §3 — `scripts/run_vision_pipeline.py`

Le script orchestre les briques `indusense.vision` dans l'ordre — il n'implémente aucun
calcul lui-même :

1. `load_train_val` / `load_test` — chargement + split (déjà dans `dataset.py`)
2. `build_pipeline` (optionnel, `--augment`) — augmentation (`augment.py`)
3. `build_autoencoder` + `mse_ssim_loss` — construction du modèle (`model.py`)
4. `OfflineEmissionsTracker` — mesure carbone **forcée** sur `config.COUNTRY_ISO_CODE`
   (pas de géolocalisation IP → même résultat en salle, hors-ligne, ou en CI)
5. `train()` — entraînement + MLflow (`train.py`)
6. `reconstruction_errors` / `calibrate_threshold` / `evaluate` — score, seuil, AUROC (`anomaly.py`)
7. `plot_confusion_matrix` / `plot_score_histogram` — figures
8. Écriture d'un rapport JSON (`reports/<run_name>_report.json`)

Arguments CLI : `--epochs`, `--img-size`, `--batch-size`, `--filters`, `--alpha-loss`,
`--augment`, `--no-mlflow`, `--no-carbon`, `--run-name`.

```bash
uv run python scripts/run_vision_pipeline.py --epochs 30 --img-size 128 --batch-size 32
```

## §4 — Exécuter le pipeline en une commande

In [2]:
import subprocess, sys, json

cmd = [
    sys.executable, "scripts/run_vision_pipeline.py",
    "--epochs", "15", "--img-size", "128", "--batch-size", "16",
    "--run-name", "tp7_demo", "--no-mlflow",
]
print("$", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True, cwd=".")
print(result.stdout[-2500:])
if result.returncode != 0:
    print("STDERR:\n", result.stderr[-3000:])
print(f"\nCode retour : {result.returncode}")


$ C:\Users\Aelion\py-init\.venv\Scripts\python.exe scripts/run_vision_pipeline.py --epochs 15 --img-size 128 --batch-size 16 --run-name tp7_demo --no-mlflow


=============== 1s 194ms/step - loss: 0.0359
 2/11 ==================== 1s 168ms/step - loss: 0.0367
 3/11 ==================== 1s 167ms/step - loss: 0.0357
 4/11 ==================== 1s 166ms/step - loss: 0.0357
 5/11 ==================== 1s 168ms/step - loss: 0.0347
 6/11 ==================== 0s 169ms/step - loss: 0.0343
 7/11 ==================== 0s 170ms/step - loss: 0.0338
 8/11 ==================== 0s 169ms/step - loss: 0.0336
 9/11 ==================== 0s 170ms/step - loss: 0.0335
10/11 ==================== 0s 170ms/step - loss: 0.0334
11/11 ==================== 0s 163ms/step - loss: 0.0333
11/11 ==================== 2s 194ms/step - loss: 0.0333 - val_loss: 0.0328 - learning_rate: 0.0010
Epoch 15/15

 1/11 ==================== 1s 189ms/step - loss: 0.0344
 2/11 ==================== 1s 168ms/step - loss: 0.0351
 3/11 ==================== 1s 169ms/step - loss: 0.0343
 4/11 ==================== 1s 169ms/step - loss: 0.0342
 5/11 ==================== 1s 172ms/step - loss: 0.0333
 6/

In [3]:
report = json.loads((config.REPORTS_DIR / "tp7_demo_report.json").read_text(encoding="utf-8"))
print(json.dumps(report, indent=2, ensure_ascii=False))


{
  "run_name": "tp7_demo",
  "args": {
    "epochs": 15,
    "img_size": 128,
    "batch_size": 16,
    "filters": [
      32,
      64,
      128,
      64
    ],
    "alpha_loss": 0.8,
    "augment": false
  },
  "n_params": 334019,
  "ratio_compression": 12.0,
  "epochs_run": 15,
  "best_val_loss": 0.032008085399866104,
  "metrics": {
    "auroc": 0.8484126984126984,
    "threshold": 0.003438673447817564,
    "tp": 25,
    "tn": 20,
    "fp": 0,
    "fn": 38,
    "recall": 0.3968253968253968,
    "precision": 1.0,
    "specificity": 1.0
  },
  "emissions_gco2eq": 0.015604995184177774,
  "energy_wh": 0.2784666961255156,
  "checkpoint": "C:\\Users\\Aelion\\py-init\\DL\\checkpoints\\tp7_demo.keras"
}


## §5 — Idempotence : deux exécutions, un seul score ?

*Point de réflexion (Étape 12 du md, transposé au DL)* : que doit renvoyer le script s'il
est relancé deux fois de suite ? Pour répondre "le même score", il faut fixer **toutes**
les sources de hasard, pas seulement le split train/val :

- `random.seed`, `np.random.seed` **et** `tf.random.set_seed` — initialisation des poids,
  ordre des batches
- Le split train/val (déjà fixé via `config.RANDOM_SEED` dans `load_train_val`)

`set_seeds()` dans le script fixe les trois. On vérifie ici que deux runs identiques
produisent un AUROC et un val_loss strictement égaux.

In [4]:
cmd2 = [
    sys.executable, "scripts/run_vision_pipeline.py",
    "--epochs", "15", "--img-size", "128", "--batch-size", "16",
    "--run-name", "tp7_demo_rerun", "--no-mlflow",
]
result2 = subprocess.run(cmd2, capture_output=True, text=True, cwd=".")
print(f"Code retour : {result2.returncode}")

report2 = json.loads((config.REPORTS_DIR / "tp7_demo_rerun_report.json").read_text(encoding="utf-8"))

for key in ["best_val_loss"]:
    v1, v2 = report[key], report2[key]
    print(f"{key:16s} run1={v1:.8f}  run2={v2:.8f}  identique={v1 == v2}")

for key in ["auroc", "threshold"]:
    v1, v2 = report["metrics"][key], report2["metrics"][key]
    print(f"{key:16s} run1={v1:.8f}  run2={v2:.8f}  identique={v1 == v2}")


Code retour : 0
best_val_loss    run1=0.03200809  run2=0.03200809  identique=True
auroc            run1=0.84841270  run2=0.84841270  identique=True
threshold        run1=0.00343867  run2=0.00343867  identique=True


## §6 — Synthèse

**Fait dans ce TP7 :**
- `config.py` centralise tous les chemins et paramètres — plus aucun chemin en dur dans
  le pipeline d'entraînement/évaluation.
- `scripts/run_vision_pipeline.py` rejoue `data → modèle → entraînement (+MLflow+CodeCarbon)
  → évaluation → figures` en une commande CLI, avec seeds fixées pour l'idempotence.
- La mesure carbone utilise `OfflineEmissionsTracker(country_iso_code="FRA")` — déterministe,
  pas de dépendance à une géolocalisation IP (contrairement au TP6 qui utilisait la
  détection automatique).

**Hors périmètre de ce TP7 (chantier ML du md, projet `ML/` séparé) :**
- Partie A — étude Optuna bornée (TPE, pruning) sur le XGBoost de maintenance.
- Partie C — SHAP sur le modèle tabulaire (summary plot, waterfall, dependence plots).
- Partie D côté ML — `src/indusense/maintenance/` et `scripts/run_maintenance_pipeline.py`.
- Partie E — tableau d'arbitrage final `modèle | PR-AUC | gCO₂eq | interprétabilité | décision`.

**Écart résiduel côté DL** : le scorer SSE de TP5 §1 (construction du `keras.Model` de
scoring pour SHAP `GradientExplainer`) reste inline dans le notebook — un candidat naturel
pour une future fonction `src/indusense/vision/explain.py`, non traité ici.